In [28]:
# Working with langchain, vector stores, and splitters
import langchain
import langchain_core
import langchain_text_splitters
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    SentenceTransformersTokenTextSplitter,
    TokenTextSplitter,
)


In [29]:
text = """langchain is a framework for developing llm appls powered by llms.
          It connects different components such as models, prompts, configuration settings """

In [30]:
text_splitter = RecursiveCharacterTextSplitter(
                chunk_size= 50,
                chunk_overlap= 10,
)

chunks = text_splitter.split_text(text)

for i in chunks:
  print(i)

langchain is a framework for developing llm appls
llm appls powered by llms.
It connects different components such
such as models, prompts, configuration settings


In [31]:
for i,chunk in enumerate(chunks):
  print(f"Chunk {i+1}: {chunk}")

Chunk 1: langchain is a framework for developing llm appls
Chunk 2: llm appls powered by llms.
Chunk 3: It connects different components such
Chunk 4: such as models, prompts, configuration settings


**HuggingFaceEmbeddings**

In [32]:
#!pip install langchain_community
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

In [33]:
model = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=model)

text_splitter = RecursiveCharacterTextSplitter(
                chunk_size= 50,
                chunk_overlap= 10,
)

e:\github\GenAI_HF_OpenAI\Hugging_Face_OpenAI_Azure\Code_Examples\.venv\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [34]:
text = """langchain is a framework for developing llm appls powered by llms
          It connects different components such as models, prompts, configuration settings """

In [35]:
# 将文本分割成 Document 对象（文档对象），而不是简单的字符串
# Document 包含两个主要属性：
# - page_content: 文本内容（chunk）
# - metadata: 元数据（如来源信息）

chunks2 = text_splitter.create_documents([text])

In [36]:
for i in chunks2:
  print(i)

page_content='langchain is a framework for developing llm appls'
page_content='llm appls powered by llms'
page_content='It connects different components such'
page_content='such as models, prompts, configuration settings'


In [37]:
vectorstore = FAISS.from_documents(chunks2, embeddings)

all_docs = list(vectorstore.docstore._dict.values())

In [38]:
for i, doc in enumerate(all_docs):
    print(f"Document {i+1}:\n{doc.page_content}\n")

Document 1:
langchain is a framework for developing llm appls

Document 2:
llm appls powered by llms

Document 3:
It connects different components such

Document 4:
such as models, prompts, configuration settings



#### example 1

In [39]:
query = "blackholes"

results = vectorstore.similarity_search(query, k=4)

#Retrieving
for i, doc in enumerate(results, 1):
    print(f"\nResult {i}:")
    print(doc.page_content)


Result 1:
It connects different components such

Result 2:
langchain is a framework for developing llm appls

Result 3:
llm appls powered by llms

Result 4:
such as models, prompts, configuration settings


In [40]:
#building retriever
retriever=vectorstore.as_retriever()

query_embedding = embeddings.embed_query("langchain")

similar_docs = vectorstore.similarity_search_by_vector(query_embedding, k=5)  # top 5 results

In [41]:
for doc in similar_docs:
    print(doc.page_content)

langchain is a framework for developing llm appls
It connects different components such
llm appls powered by llms
such as models, prompts, configuration settings


#### example 2

In [42]:
query_embedding = embeddings.embed_query("Fishing boat")

similar_docs = vectorstore.similarity_search_by_vector(query_embedding, k=2)  # top 5 results

for doc in similar_docs:
    print(doc.page_content)

It connects different components such
langchain is a framework for developing llm appls


What happened above and why it still shows results

What you’re seeing is actually expected behavior for most vector stores, not necessarily a bug.
Why you still get results for unrelated queries
>> Vector similarity search doesn’t work like keyword search. 
   It always returns the “closest” vectors, even if they are not actually relevant.

For example>>
You ask: “apple”
Your database contains only chunks about cars, physics, and finance
The system still returns something like “car engine” — because it’s the least dissimilar, 
even if it’s still very far away semantically
There is no built-in concept of “no match” unless you explicitly enforce it.


A few common issues cause this:

1. No similarity threshold
Most vector stores (like FAISS, Chroma, Pinecone, etc.) default to: “Give me top K nearest neighbors”
They do not check: “Are these neighbors actually similar enough?”
So even irrelevant chunks get returned.

2. Distance vs similarity misunderstanding
Depending on your setup:
Cosine similarity → higher = better
Euclidean distance → lower = better
If you’re not checking the score properly, you may treat weak matches as valid ones.

3. Embeddings are never truly “zero-related”
Embedding models (like OpenAI embeddings) map everything into the same vector space.
So even unrelated words:
“banana” and “quantum mechanics”
…will still have some measurable similarity.

4. Chunking issues
If your chunks are:
too large → they contain mixed topics
too small → lack enough context
Then embeddings become noisy and matches degrade.


In [43]:
#Option 1
#Apply a similarity threshold, After retrieving results, filter them:
query_embedding = embeddings.embed_query("Fishing boat")
results = vectorstore.similarity_search_with_score_by_vector(query_embedding, k=5)

for doc, score in results:
    print(score, doc.page_content)

1.9245224 It connects different components such
2.0228248 langchain is a framework for developing llm appls
2.0596392 such as models, prompts, configuration settings
2.0717309 llm appls powered by llms


In [44]:
filtered = [doc for doc, score in results if score < 1.92]
for doc in filtered:
    print(doc.page_content)
#Shows nothing

In [45]:
#Apply a similarity threshold, After retrieving results, filter them:
query_embedding = embeddings.embed_query("Langchain")
results = vectorstore.similarity_search_with_score_by_vector(query_embedding, k=5)

for doc, score in results:
    print(score, doc.page_content)

0.93989617 langchain is a framework for developing llm appls
1.7672547 It connects different components such
1.8723807 llm appls powered by llms
1.9892986 such as models, prompts, configuration settings


In [46]:
filtered = [doc for doc, score in results if score < 2]
for doc in filtered:
    print(doc.page_content)

langchain is a framework for developing llm appls
It connects different components such
llm appls powered by llms
such as models, prompts, configuration settings


But we shouldnt be changing threshold
Here we are using distance metric (FAISS with L2 distance)
 ~0.9 → very close (highly relevant)
 ~1.5–1.9 → moderately related
 ~2.0+ → weak / irrelevant
>"Fishing boat" : Scores: ~1.92 – 2.07 (All chunks are far away in vector space, so nothing passes)
>"Langchain" : Scores: 0.93   ← strong match
                        1.76
                        1.87
                        1.98
First result is very relevant, Others are loosely related but still closer than random, so with < 2 we get everything.
So good threshold could be between 0.93 and 1.92 ,threshold ≈ 1.3 or 1.4


In [47]:
threshold = 1.4
filtered = [doc for doc, score in results if score < threshold]
for doc in filtered:
    print(doc.page_content)

langchain is a framework for developing llm appls


#### Vector search always returns results — relevance is OUR responsibility

In [48]:
#Using similarity search
#If not using scores to filter by relevance
#only when you just want top-k results
"""Internally calls embeddings.embed_query()
Returns only documents (no scores)"""
results_new = vectorstore.similarity_search("Langchain", k=5)

for doc in results_new:
    print(doc.page_content)

langchain is a framework for developing llm appls
It connects different components such
llm appls powered by llms
such as models, prompts, configuration settings


In [49]:
#Using similarity_search_with_score
"""
Handles embedding internally, Returns (doc, score) pairs, Score is usually distance (lower = better)
"""
results_new2 = vectorstore.similarity_search_with_score("Langchain", k=5)

for doc, score in results_new2:
    print(score, doc.page_content)

threshold = 1.4  # based on our earlier observation

filtered = [
    doc for doc, score in results
    if score < threshold
]

for doc in filtered:
    print(doc.page_content)

    

0.93989617 langchain is a framework for developing llm appls
1.7672547 It connects different components such
1.8723807 llm appls powered by llms
1.9892986 such as models, prompts, configuration settings
langchain is a framework for developing llm appls


In [50]:
#Using similarity_search_with_relevance_scores
"""
Here Scores are normalized → 0 to 1, Higher = better (unlike distance)
"""
results = vectorstore.similarity_search_with_relevance_scores(query, k=5)
threshold = 0.7
filtered = [doc for doc, score in results if score > threshold]

if not filtered:
    print("No relevant results found")
else:
    for doc in filtered:
        print(doc.page_content)

No relevant results found


C:\Users\Mason\AppData\Local\Temp\ipykernel_9912\3294070310.py:5: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='db250926-99c8-4bed-8a19-5b3e49e2ca36', metadata={}, page_content='It connects different components such'), -0.30028194859735513), (Document(id='6e766ed5-5cd4-4efa-bfdd-ca437eccf051', metadata={}, page_content='langchain is a framework for developing llm appls'), -0.37241139078932295), (Document(id='5a9ded6c-f15b-4b56-94f1-1556f17877a7', metadata={}, page_content='llm appls powered by llms'), -0.40772294770241735), (Document(id='cc735264-4515-4291-8823-aa0c01e98d94', metadata={}, page_content='such as models, prompts, configuration settings'), -0.5052841354606468)]
  results = vectorstore.similarity_search_with_relevance_scores(query, k=5)
